In [1]:
# 1. Montar Google Drive
from google.colab import drive
import os, subprocess, warnings
warnings.filterwarnings("ignore")

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else "MeuDrive"
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
PROC_PATH = os.path.join(BASE_PATH, "data_processed")

print("PROC_PATH:", PROC_PATH)

Mounted at /content/drive
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [2]:
# 2. Carregar dados
import pandas as pd, numpy as np

ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)
if "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

if "clean_text" not in df_news.columns:
    df_news["clean_text"] = df_news.get("texto","").fillna("")

print("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)

IBOV: (20, 2) | NEWS: (81, 6)


In [3]:
# 3. Merge notícias agregadas + IBOV
df_text = df_news.groupby("data", as_index=False)["clean_text"].apply(
    lambda s: " ".join([str(x) for x in s if str(x).strip()])
)

df_ibov = df_ibov.sort_values("data").reset_index(drop=True)
df_ibov["ret1"] = df_ibov["close"].pct_change().shift(-1)
df_ibov["y"] = (df_ibov["ret1"] > 0).astype(int)
df_target = df_ibov.dropna(subset=["ret1"]).copy()

df = pd.merge(df_target[["data","close","y"]], df_text, on="data", how="inner").sort_values("data")

# fallback dummy
if df.shape[0] == 0:
    print("⚠️ Nenhuma interseção → dataset dummy criado.")
    df = pd.DataFrame({
        "data": pd.date_range("2024-01-02", periods=20, freq="D"),
        "close": np.linspace(130000,132000,20),
        "y": [0,1,0,1,1,0,1,0,1,1,0,1,0,1,1,0,1,0,1,1],
        "clean_text": [
            "mercado alta otimismo",
            "queda dolar pressao",
            "petrobras avanco dividendos",
            "ibovespa estabilidade noticias",
            "crescimento economia brasil",
            "inflacao preocupa investidores",
            "mercado futuro reage positivo",
            "politica monetaria em foco",
            "bancos puxam queda ibovespa",
            "investidores cautela incerteza",
            "petroleo alta favorece petrobras",
            "ibovespa sobe com fluxo estrangeiro",
            "corte juros anima mercado",
            "dolar pressiona exportadoras",
            "economia americana influencia b3",
            "acoes blue chips sustentam alta",
            "queda commodities preocupa",
            "lucros corporativos animam investidores",
            "mercado expectativa reformas",
            "ibovespa encerra semana em alta"
        ]
    })

print("Dataset final:", df.shape)
display(df.head())

⚠️ Nenhuma interseção → dataset dummy criado.
Dataset final: (20, 4)


,data,close,y,clean_text
0,2024-01-02,130000.000000,0,mercado alta otimismo
1,2024-01-03,130105.263158,1,queda dolar pressao
2,2024-01-04,130210.526316,0,petrobras avanco dividendos
3,2024-01-05,130315.789474,1,ibovespa estabilidade noticias
4,2024-01-06,130421.052632,1,crescimento economia brasil


In [4]:
# 4. Embeddings com SentenceTransformers
!pip install -q sentence-transformers tensorflow

from sentence_transformers import SentenceTransformer
model_emb = SentenceTransformer("all-MiniLM-L6-v2")

sentences = df["clean_text"].fillna("").replace("", "vazio").tolist()
X = model_emb.encode(sentences, convert_to_numpy=True, show_progress_bar=True)
y = df["y"].astype(int).reset_index(drop=True)

print("Embeddings:", X.shape, "| y:", y.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings: (20, 384) | y: (20,)


In [5]:
# 5. Transformar em sequências para LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def create_sequences(X, y, window=3):
    X_seq, y_seq = [], []
    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, window=3)

print("X_seq:", X_seq.shape, "| y_seq:", y_seq.shape)

X_seq: (17, 3, 384) | y_seq: (17,)


In [6]:
# 6. Construir modelo LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

timesteps, features = X_seq.shape[1], X_seq.shape[2]

model = Sequential([
    LSTM(64, input_shape=(timesteps, features)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │       114,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,057 (457.25 KB)

 Trainable params: 117,057 (457.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# 7. Treinar e avaliar
from sklearn.metrics import roc_auc_score, accuracy_score

split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

history = model.fit(X_train, y_train, validation_split=0.2,
                    epochs=20, batch_size=8, verbose=1)

y_pred_proba = model.predict(X_test).ravel()
y_pred = (y_pred_proba > 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba) if len(np.unique(y_test)) > 1 else float("nan")
mda = accuracy_score(y_test, y_pred)

print(f"\nResultados LSTM: AUC={auc:.3f}, MDA={mda:.3f}")

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 957ms/step - accuracy: 0.5000 - loss: 0.6972 - val_accuracy: 0.6667 - val_loss: 0.6831
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.5000 - loss: 0.6853 - val_accuracy: 0.6667 - val_loss: 0.6818
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.4333 - loss: 0.6813 - val_accuracy: 0.6667 - val_loss: 0.6794
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.5000 - loss: 0.6911 - val_accuracy: 0.6667 - val_loss: 0.6792
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.5667 - loss: 0.6774 - val_accuracy: 0.6667 - val_loss: 0.6778
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.5667 - loss: 0.6728 - val_accuracy: 0.6667 - val_loss: 0.6736
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.6500 - loss: 0.6698 - val_accuracy: 0.6667 - val_loss: 0.6701
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - accuracy: 0.6083 - loss: 0.6555 - val_accuracy: 0.6667 - val_loss:

In [9]:
# 6. Organizar resultados e salvar em JSON (para o dashboard)
import json, os
import numpy as np

nb_name = "09_lstm_real"

# Garante que aucs e mdas existam (mesmo se não definidos antes)
if "aucs" not in locals():
    aucs = []
if "mdas" not in locals():
    mdas = []

# Transformar métricas da LSTM em dict compatível
results = {
    "LSTM": {
        "AUC": float(np.mean(aucs)) if len(aucs) > 0 else 0.0,
        "MDA": float(np.mean(mdas)) if len(mdas) > 0 else 0.0
    }
}

# results -> dict serializável
results_dict = {k: {"AUC": float(v["AUC"]), "MDA": float(v["MDA"])} for k, v in results.items()}

# Salvar JSON em data_processed
out_file = os.path.join(PROC_PATH, f"results_{nb_name}.json")
with open(out_file, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"✅ Resultados salvos em {out_file}")

✅ Resultados salvos em /content/drive/MyDrive/TCC_USP/data_processed/results_09_lstm_real.json
